# कॉल ग्राफ विश्लेषण और दृढ़ता से जुड़े घटक (SCC) पहचान

यह नोटबुक UnifyWeaver की उन्नत कोड विश्लेषण क्षमताओं की पड़ताल करती है:

- **कॉल ग्राफ निर्माण** — Prolog कोड से निर्भरता ग्राफ बनाना
- **SCC पहचान** — दृढ़ता से जुड़े घटकों की खोज (म्यूचुअल रिकर्सन)
- **पैटर्न विश्लेषण** — रिकर्सन पैटर्न को समझना
- **निर्भरता विज़ुअलाइज़ेशन** — प्रेडिकेट संबंधों का दृश्य चित्रण

## सीखने के उद्देश्य

- समझना कि UnifyWeaver कोड संरचना का विश्लेषण कैसे करता है
- कॉल ग्राफ बनाना और उनका निरीक्षण करना
- तर्जन (Tarjan) एल्गोरिथ्म का उपयोग करके म्यूचुअल रिकर्सन का पता लगाना
- कोड निर्भरता की कल्पना करना

## सेटअप

UnifyWeaver और विश्लेषण मॉड्यूल लोड करें।

In [ ]:
% आरंभीकरण लोड करें
['../init'].

% विश्लेषण मॉड्यूल लोड करें
use_module(unifyweaver(core/advanced/call_graph)).
use_module(unifyweaver(core/advanced/scc_detection)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## उदाहरण 1: सरल कॉल ग्राफ

आइए एक साधारण प्रेडिकेट से शुरू करें और उसका कॉल ग्राफ बनाएं।

In [ ]:
% ancestor प्रेडिकेट परिभाषित करें
:- dynamic ancestor/2.
:- dynamic parent/2.

% parent तथ्य
parent(abraham, isaac).
parent(isaac, jacob).

% ancestor नियम
ancestor(X, Y) :- parent(X, Y).
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

### कॉल ग्राफ बनाएं

In [ ]:
% ancestor के लिए कॉल ग्राफ बनाएं
build_call_graph([ancestor/2], _Graph),
format('Call Graph for ancestor/2:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### निर्भरता का विश्लेषण करें

In [ ]:
% ancestor/2 की सभी निर्भरताएं प्राप्त करें
get_dependencies(ancestor/2, _Deps),
format('Dependencies of ancestor/2: ~w~n', [_Deps]).

% जांचें कि क्या यह स्व-पुनरावर्ती है
(is_self_recursive(ancestor/2) ->
    writeln('✓ ancestor/2 is self-recursive')
;
    writeln('✗ ancestor/2 is not self-recursive')
).

## उदाहरण 2: म्यूचुअल रिकर्सन पहचान

अब सम/विषम उदाहरण के साथ म्यूचुअल रिकर्सन का पता लगाते हैं।

In [ ]:
% परस्पर पुनरावर्ती प्रेडिकेट्स परिभाषित करें
:- dynamic is_even/1.
:- dynamic is_odd/1.

is_even(0).
is_even(N) :- N > 0, N1 is N - 1, is_odd(N1).

is_odd(1).
is_odd(N) :- N > 1, N1 is N - 1, is_even(N1).

### दोनों प्रेडिकेट्स के लिए कॉल ग्राफ बनाएं

In [ ]:
% दोनों प्रेडिकेट्स के लिए कॉल ग्राफ बनाएं
build_call_graph([is_even/1, is_odd/1], _Graph),
format('Call Graph for is_even/1 and is_odd/1:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### दृढ़ता से जुड़े घटक (SCC) खोजें

In [ ]:
% ग्राफ को फिर से बनाएं क्योंकि चर नोटबुक सेल के बीच बने नहीं रहते हैं
build_call_graph([is_even/1, is_odd/1], _Graph),
% तर्जन एल्गोरिथ्म का उपयोग करके SCC खोजें
find_sccs(_Graph, _SCCs),
format('Strongly Connected Components:~n'),
forall(member(_SCC, _SCCs),
    format('  ~w~n', [_SCC])).

### जांचें कि क्या SCC तुच्छ (Trivial) है

In [ ]:
% व्युत्पन्न मानों को फिर से बनाएं ताकि यह सेल भी स्वतंत्र रूप से चले
build_call_graph([is_even/1, is_odd/1], _Graph),
find_sccs(_Graph, _SCCs),
% प्रत्येक SCC की जांच करें
forall(member(_SCC, _SCCs),
    (   is_trivial_scc(_SCC) ->
        format('  ~w is trivial (single predicate)~n', [_SCC])
    ;
        format('  ~w is NON-TRIVIAL (mutual recursion!)~n', [_SCC])
    )
).

## उदाहरण 3: जटिल कॉल ग्राफ

आइए कई प्रेडिकेट्स वाली अधिक जटिल प्रणाली का विश्लेषण करें।

In [ ]:
% एकाधिक प्रेडिकेट्स वाला एक छोटा प्रोग्राम परिभाषित करें
:- dynamic grandparent/2.
:- dynamic sibling/2.
:- dynamic cousin/2.

% grandparent, parent का उपयोग करता है
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

% sibling: समान माता-पिता, भिन्न बच्चे
sibling(X, Y) :- parent(P, X), parent(P, Y), X \= Y.

% cousin: माता-पिता भाई-बहन हैं
cousin(X, Y) :- parent(P1, X), parent(P2, Y), sibling(P1, P2).

### पूर्ण कॉल ग्राफ बनाएं

In [ ]:
% सभी प्रेडिकेट्स के लिए कॉल ग्राफ बनाएं
build_call_graph([grandparent/2, sibling/2, cousin/2], _Graph),
format('Complete Call Graph:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### प्रेडिकेट समूह खोजें

आरंभिक प्रेडिकेट वाले परस्पर पुनरावर्ती प्रेडिकेट समूह को खोजना।

In [ ]:
% cousin/2 युक्त परस्पर पुनरावर्ती समूह खोजें
predicates_in_group(cousin/2, _Group),
format('Mutually recursive group containing cousin/2: ~w~n', [_Group]).

## उदाहरण 4: पैटर्न पहचान

रिकर्सन प्रकारों का विश्लेषण करने के लिए पैटर्न मैचर का उपयोग करें।

In [ ]:
% विभिन्न पुनरावर्ती पैटर्न परिभाषित करें
:- dynamic count/3.     % टेल रिकर्सिव
:- dynamic factorial/2. % रैखिक रिकर्सिव
:- dynamic fib/2.       % ट्री रिकर्सिव (या यदि पहचाना गया तो रैखिक)

% टेल रिकर्सिव count
count([], Acc, Acc).
count([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count(T, Acc1, N).

% रैखिक रिकर्सिव factorial
factorial(0, 1).
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),
    F is N * F1.

% फाइबोनैचि (रैखिक या ट्री के रूप में पहचाना जा सकता है)
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.

### टेल रिकर्सन का पता लगाएं

In [ ]:
% जांचें कि क्या count/3 टेल रिकर्सिव है
(is_tail_recursive_accumulator(count/3, _AccInfo) ->
    format('✓ count/3 is tail recursive: ~w~n', [_AccInfo])
;
    writeln('✗ count/3 is not tail recursive')
).

### रैखिक रिकर्सन का पता लगाएं

In [ ]:
% जांचें कि क्या factorial/2 रैखिक रिकर्सिव है
(is_linear_recursive_streamable(factorial/2) ->
    writeln('✓ factorial/2 is linear recursive')
;
    writeln('✗ factorial/2 is not linear recursive')
).

### पुनरावर्ती कॉल की गणना करें

In [ ]:
% फाइबोनैचि में पुनरावर्ती कॉल की गणना करें
functor(_FibHead, fib, 2),
user:clause(_FibHead, _FibBody),
once(contains_call_to(_FibBody, fib)),
count_recursive_calls(_FibBody, fib, _Count),
format('Fibonacci body has ~w recursive calls~n', [_Count]).

## DOT प्रारूप के साथ विज़ुअलाइज़ेशन

आइए अपने कॉल ग्राफ का एक Graphviz DOT प्रतिनिधित्व उत्पन्न करें।

In [ ]:
% DOT प्रारूप उत्पन्न करने के लिए सहायक
generate_dot(Graph, DotCode) :-
    findall(Line,
        (   member(From -> To, Graph),
            format(atom(Line), '  "~w" -> "~w";', [From, To])
        ),
        Lines),
    atomic_list_concat(['digraph CallGraph {', '  rankdir=LR;' | Lines], '\n', Body),
    format(atom(DotCode), '~w~n}~n', [Body]).

% सम/विषम ग्राफ के लिए DOT उत्पन्न करें
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
writeln('DOT Code for even/odd call graph:'),
writeln(_DotCode).

### DOT फ़ाइल सहेजें

In [ ]:
% DOT स्रोत को फिर से बनाएं क्योंकि चर सेल के बीच बने नहीं रहते हैं
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
setup_call_cleanup(
    open('/shared/data/even_odd_graph.dot', write, _Stream),
    write(_Stream, _DotCode),
    close(_Stream)),
writeln('✓ Saved to /shared/data/even_odd_graph.dot').

writeln('To visualize, run:'),
writeln('  dot -Tpng /shared/data/even_odd_graph.dot -o even_odd_graph.png').

## अभ्यास: अपने स्वयं के कोड का विश्लेषण करें

अपने स्वयं के प्रेडिकेट्स को परिभाषित करने और उनका विश्लेषण करने का प्रयास करें!

In [ ]:
% अपने प्रेडिकेट्स यहाँ परिभाषित करें
% फिर कॉल ग्राफ बनाएं, SCC खोजें और पैटर्न का पता लगाएं

% उदाहरण:
% :- dynamic my_predicate/2.
% my_predicate(...) :- ...

% build_call_graph([my_predicate/2], Graph).


## सारांश

इस नोटबुक में, आपने सीखा:

✅ Prolog कोड से कॉल ग्राफ कैसे बनाएं

✅ म्यूचुअल रिकर्सन के लिए दृढ़ता से जुड़े घटकों (SCC) का पता कैसे लगाएं

✅ रिकर्सन प्रकारों को वर्गीकृत करने के लिए पैटर्न मैचर का उपयोग कैसे करें

✅ प्रेडिकेट निर्भरता का विश्लेषण कैसे करें

✅ DOT प्रारूप के साथ कॉल ग्राफ को कैसे विज़ुअलाइज़ करें

## उन्नत विषय

अधिक उन्नत विश्लेषण के लिए:

- **टोपोलॉजिकल सॉर्टिंग**: निर्भरता के आधार पर SCC को क्रमबद्ध करने के लिए `topological_order/2` का उपयोग करें
- **कस्टम पैटर्न मैचर**: अपने स्वयं के पैटर्न पहचान प्रेडिकेट लिखें
- **संचायक पैटर्न निष्कर्षण**: विस्तृत विश्लेषण के लिए `extract_accumulator_pattern/2` का उपयोग करें
- **रैखिक रिकर्सन को प्रतिबंधित करें**: विभिन्न संकलन रणनीतियों को बाध्य करने के लिए `forbid_linear_recursion/1` का उपयोग करें

## संदर्भ और संबंधित फ़ाइलें

- अध्याय 10: Prolog आत्मनिरीक्षण और सिद्धांत
- `src/unifyweaver/core/advanced/call_graph.pl`
- `src/unifyweaver/core/advanced/scc_detection.pl`
- `src/unifyweaver/core/advanced/pattern_matchers.pl`